In [1]:
import sys
sys.path.append('../../')

In [2]:
import pandas as pd
import numpy as np
from tabulate import tabulate
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, PowerTransformer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, chi2, f_classif
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from src.utils import helpers as func
import prince as pr
import plotly.express as px

In [15]:
path = '../../data/processed/df_acumulado.parquet'
df = func.load_data(path)

Cargando datos desde: ../../data/processed/df_acumulado.parquet
Datos cargados exitosamente.
Dimensiones del DataFrame: (573, 11)

Primeras filas del DataFrame:


,fecha,valor_acumulado,semana,rolling_mean,rolling_std,upper_bound,lower_bound,is_outlier,valor_acumulado_corregido,mes,año
0,2014-01-06,131.0,2,NaN,NaN,NaN,NaN,False,131.0,1,2014
1,2014-01-13,789.0,3,NaN,NaN,NaN,NaN,False,789.0,1,2014
2,2014-01-20,1722.0,4,880.666667,799.451270,1999.898445,-238.565112,False,1722.0,1,2014
3,2014-01-27,2745.0,5,1346.750000,1137.987807,2939.932930,-246.432930,False,2745.0,1,2014
4,2014-02-03,3896.0,6,2288.000000,1336.895658,4159.653921,416.346079,False,3896.0,2,2014


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 573 entries, 0 to 572
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   fecha                      573 non-null    datetime64[ns]
 1   valor_acumulado            573 non-null    float64       
 2   semana                     573 non-null    UInt32        
 3   rolling_mean               571 non-null    float64       
 4   rolling_std                571 non-null    float64       
 5   upper_bound                571 non-null    float64       
 6   lower_bound                571 non-null    float64       
 7   is_outlier                 573 non-null    bool          
 8   valor_acumulado_corregido  573 non-null    float64       
 9   mes                        573 non-null    int32         
 10  año                        573 non-null    int32         
dtypes: UInt32(1), bool(1), datetime64[ns](1), float64(6), int32(2)
memory u

None

In [16]:
df = func.clean_column_names(df)
df = df[['fecha', 'valor_acumulado_corregido']]
df['valor'] = df['valor_acumulado_corregido'].diff().fillna(0)
df.columns = ['fecha', 'valor_acumulado', 'valor']
df.iloc[0, df.columns.get_loc('valor')] = 131
df["año"] = df["fecha"].dt.isocalendar().year
df["semana"] = df["fecha"].dt.isocalendar().week
df['mes'] = df['fecha'].dt.month
df['trimestre'] = df['fecha'].dt.quarter
df['semestre'] = df['fecha'].dt.month.apply(lambda m: 1 if m <= 6 else 2)
df['temporada'] = df['mes'].apply(func.asignar_temporada)
df[['valor', 'valor_acumulado']] = df[['valor', 'valor_acumulado']].astype(int)
df.head()

,fecha,valor_acumulado,valor,año,semana,mes,trimestre,semestre,temporada
0,2014-01-06,131,131,2014,2,1,1,1,Invierno
1,2014-01-13,789,658,2014,3,1,1,1,Invierno
2,2014-01-20,1722,933,2014,4,1,1,1,Invierno
3,2014-01-27,2745,1023,2014,5,1,1,1,Invierno
4,2014-02-03,3896,1151,2014,6,2,1,1,Invierno


In [17]:
sem_año = 52
lag_años = [1,2]
grupo_cols = ['fecha']

df.sort_values(grupo_cols, inplace=True)

for año in lag_años:
    # lag año con respecto al anterior año indicado
    df[f"lag_{año}y"] = df["año"].shift(sem_año * año).fillna(0).astype(int)
    # lag del valor con respecto al año anterior indicado
    df[f"lag_v{año}y"] = df["valor"].shift(sem_año * año).fillna(0).astype(int)
    # lag del valor acumulado con respecto al año anterior indicado
    df[f"lag_va{año}y"] = df["valor_acumulado"].shift(sem_año * año).fillna(0).astype(int)
df

,fecha,valor_acumulado,valor,año,semana,mes,trimestre,semestre,temporada,lag_1y,lag_v1y,lag_va1y,lag_2y,lag_v2y,lag_va2y
0,2014-01-06,131,131,2014,2,1,1,1,Invierno,0,0,0,0,0,0
1,2014-01-13,789,658,2014,3,1,1,1,Invierno,0,0,0,0,0,0
2,2014-01-20,1722,933,2014,4,1,1,1,Invierno,0,0,0,0,0,0
3,2014-01-27,2745,1023,2014,5,1,1,1,Invierno,0,0,0,0,0,0
4,2014-02-03,3896,1151,2014,6,2,1,1,Invierno,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
568,2024-11-25,1219814,2500,2024,48,11,4,2,Otoño,2023,2586,1075184,2022,2427,924786
569,2024-12-02,1222718,2904,2024,49,12,4,2,Invierno,2023,2914,1078098,2022,2514,927300
570,2024-12-09,1225400,2682,2024,50,12,4,2,Invierno,2023,2737,1080835,2022,2223,929523
571,2024-12-16,1227959,2559,2024,51,12,4,2,Invierno,2023,2249,1083084,2022,2196,931719


In [18]:
df["mes_sin"] = np.sin(2 * np.pi * df["mes"] / 12)
df["mes_cos"] = np.cos(2 * np.pi * df["mes"] / 12)

In [19]:
covid_start = pd.to_datetime('2020-03-01')
# covid_end = pd.to_datetime('2022-03-31')
covid_end = pd.to_datetime('2023-05-05')

df['periodo_covid'] = pd.cut(
    df['fecha'],
    bins=[pd.Timestamp.min, covid_start, 
          covid_end, pd.Timestamp.max],
    labels=['Pre-Covid', 'Covid', 'Post-Covid'],
    include_lowest=True
)

df['festivo'] = df.apply(func.encontrar_festivos_en_semana, axis=1)
df

,fecha,valor_acumulado,valor,año,semana,mes,trimestre,semestre,temporada,lag_1y,lag_v1y,lag_va1y,lag_2y,lag_v2y,lag_va2y,mes_sin,mes_cos,periodo_covid,festivo
0,2014-01-06,131,131,2014,2,1,1,1,Invierno,0,0,0,0,0,0,5.000000e-01,0.866025,Pre-Covid,0
1,2014-01-13,789,658,2014,3,1,1,1,Invierno,0,0,0,0,0,0,5.000000e-01,0.866025,Pre-Covid,0
2,2014-01-20,1722,933,2014,4,1,1,1,Invierno,0,0,0,0,0,0,5.000000e-01,0.866025,Pre-Covid,0
3,2014-01-27,2745,1023,2014,5,1,1,1,Invierno,0,0,0,0,0,0,5.000000e-01,0.866025,Pre-Covid,0
4,2014-02-03,3896,1151,2014,6,2,1,1,Invierno,0,0,0,0,0,0,8.660254e-01,0.500000,Pre-Covid,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
568,2024-11-25,1219814,2500,2024,48,11,4,2,Otoño,2023,2586,1075184,2022,2427,924786,-5.000000e-01,0.866025,Post-Covid,0
569,2024-12-02,1222718,2904,2024,49,12,4,2,Invierno,2023,2914,1078098,2022,2514,927300,-2.449294e-16,1.000000,Post-Covid,0
570,2024-12-09,1225400,2682,2024,50,12,4,2,Invierno,2023,2737,1080835,2022,2223,929523,-2.449294e-16,1.000000,Post-Covid,0
571,2024-12-16,1227959,2559,2024,51,12,4,2,Invierno,2023,2249,1083084,2022,2196,931719,-2.449294e-16,1.000000,Post-Covid,0


In [20]:
df[['semana', 'mes', 'trimestre', 'semestre']] = df[['semana', 'mes', 'trimestre', 'semestre']].astype(object)

In [21]:
columnas_ohe = ['semana','mes','trimestre','semestre','periodo_covid','temporada']
dummies = pd.get_dummies(df[columnas_ohe], drop_first=True)
# Remove columns that overlap before joining
# dummies = dummies.drop(columns=['semana', 'mes', 'trimestre', 'semestre'])
df = df.join(dummies)
df.drop(columns=columnas_ohe, inplace=True)

In [22]:
df = func.clean_column_names(df)
df.columns = df.columns.str.replace('_covid_', '_', n=1) # Simplifica nombres de columnas

In [23]:
df.head()

,fecha,valor_acumulado,valor,año,lag_1y,lag_v1y,lag_va1y,lag_2y,lag_v2y,lag_va2y,...,mes_12,trimestre_2,trimestre_3,trimestre_4,semestre_2,periodo_covid,periodo_post-covid,temporada_otoño,temporada_primavera,temporada_verano
0,2014-01-06,131,131,2014,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
1,2014-01-13,789,658,2014,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
2,2014-01-20,1722,933,2014,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
3,2014-01-27,2745,1023,2014,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
4,2014-02-03,3896,1151,2014,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
